# Библиотеки

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
spark = SparkSession.builder \
    .appName("pyspark_als_demo") \
    .master("local[*]") \
    .getOrCreate()

In [4]:
ratings = spark.read.csv("work/data/Ratings_1.csv", header=True, inferSchema=True)
ratings.show(5)

+-------+----------+-----------+
|User-ID|      ISBN|Book-Rating|
+-------+----------+-----------+
| 276725|034545104X|          0|
| 276726|0155061224|          5|
| 276727|0446520802|          0|
| 276729|052165615X|          3|
| 276729|0521795028|          6|
+-------+----------+-----------+
only showing top 5 rows



In [5]:
users = spark.read.csv("work/data/Users.csv", header=True, inferSchema=True)
users.show(5)

+-------+--------------------+----+
|User-ID|            Location| Age|
+-------+--------------------+----+
|      1|  nyc, new york, usa|NULL|
|      2|stockton, califor...|18.0|
|      3|moscow, yukon ter...|NULL|
|      4|porto, v.n.gaia, ...|17.0|
|      5|farnborough, hant...|NULL|
+-------+--------------------+----+
only showing top 5 rows



In [6]:
books = spark.read.csv("work/data/Books.csv", header=True, inferSchema=True)
books.show(5)

+----------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+
|      ISBN|          Book-Title|         Book-Author|Year-Of-Publication|           Publisher|         Image-URL-S|         Image-URL-M|         Image-URL-L|
+----------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+
|0195153448| Classical Mythology|  Mark P. O. Morford|               2002|Oxford University...|http://images.ama...|http://images.ama...|http://images.ama...|
|0002005018|        Clara Callan|Richard Bruce Wright|               2001|HarperFlamingo Ca...|http://images.ama...|http://images.ama...|http://images.ama...|
|0060973129|Decision in Normandy|        Carlo D'Este|               1991|     HarperPerennial|http://images.ama...|http://images.ama...|http://images.ama...|
|0374157065|Flu: The Story of...|    Gina Bari

# Базовая эвристика

Popularity(book) = число оценок книги

In [7]:
book_popularity = (
    ratings
    .groupBy("ISBN")
    .agg(
        F.countDistinct("User-ID").alias("users_count"),
        F.avg("Book-Rating").alias("avg_rating")
    )
)

In [8]:
top10 = (
    book_popularity
    .join(
        books.select(
            "ISBN",
            "Book-Title",
            "Book-Author"
        ),
        on="ISBN",
        how="left"
    )
    .filter(F.col("Book-Title").isNotNull())
    .select(
        "ISBN",
        "Book-Title",
        "Book-Author",
        "users_count",
        "avg_rating"
    )
    .orderBy(
        F.desc("users_count"),
        F.desc("avg_rating")
    )
    .limit(10)
)

In [9]:
top10.show(10, truncate=False)

+----------+--------------------------------------------------+--------------+-----------+------------------+
|ISBN      |Book-Title                                        |Book-Author   |users_count|avg_rating        |
+----------+--------------------------------------------------+--------------+-----------+------------------+
|0971880107|Wild Animus                                       |Rich Shapero  |2502       |1.0195843325339728|
|0316666343|The Lovely Bones: A Novel                         |Alice Sebold  |1295       |4.468725868725869 |
|0385504209|The Da Vinci Code                                 |Dan Brown     |883        |4.652321630804077 |
|0060928336|Divine Secrets of the Ya-Ya Sisterhood: A Novel   |Rebecca Wells |732        |3.448087431693989 |
|0312195516|The Red Tent (Bestselling Backlist)               |Anita Diamant |723        |4.334716459197787 |
|044023722X|A Painted House                                   |John Grisham  |647        |3.187017001545595 |
|014200174

Главная проблема — такая система не персонализирована. Все пользователи получают один и тот же топ-10 независимо от их вкусов.

# Коллаборативная фильтрация

Только явные оценки

In [10]:
explicit = ratings.filter(F.col("Book-Rating") > 0)

Выбираем самого активного пользователя

In [11]:
target_user = (
    explicit
    .groupBy("User-ID")
    .agg(F.countDistinct("ISBN").alias("ratings_count"))
    .orderBy(F.desc("ratings_count"))
    .first()["User-ID"]
)

In [12]:
print("Выбранный пользователь:", target_user)

Выбранный пользователь: 11676


Его оценки

In [13]:
target_ratings = (
    explicit
    .filter(F.col("User-ID") == target_user)
    .select(
        "ISBN",
        F.col("Book-Rating").alias("target_rating")
    )
)

Находим общие оценки с другими пользователями

In [14]:
common_ratings = (
    explicit.alias("r")
    .join(target_ratings.alias("t"), on="ISBN")
    .filter(F.col("r.User-ID") != target_user)
    .select(
        F.col("r.User-ID").alias("other_user"),
        "ISBN",
        F.col("r.Book-Rating").alias("other_rating"),
        "target_rating"
    )
)

Сходство пользователей

In [15]:
similarity = (
    common_ratings
    .groupBy("other_user")
    .agg(
        F.count("*").alias("common_books"),
        F.avg(
            F.abs(
                F.col("other_rating") - F.col("target_rating")
            )
        ).alias("mae")
    )
)

Требуем достаточное количество общих книг

In [16]:
top3_neighbors = (
    similarity
    .filter(F.col("common_books") >= 10)
    .orderBy(
        F.asc("mae"),
        F.desc("common_books")
    )
    .limit(3)
)

In [17]:
print("Топ-3 похожих пользователя:")
top3_neighbors.show()

Топ-3 похожих пользователя:
+----------+------------+------------------+
|other_user|common_books|               mae|
+----------+------------+------------------+
|    203075|          10|               0.7|
|     95173|          10|               0.8|
|    196052|          11|0.8181818181818182|
+----------+------------+------------------+



Книги target-пользователя

In [18]:
target_books = (
    explicit
    .filter(F.col("User-ID") == target_user)
    .select("ISBN")
    .distinct()
)

Книги, высоко оценённые соседями

In [19]:
neighbors = top3_neighbors.select(
    F.col("other_user").alias("User-ID")
)

In [20]:
candidate_books = (
    explicit
    .join(neighbors, on="User-ID")
    .filter(F.col("Book-Rating") >= 8)
    .join(target_books, on="ISBN", how="left_anti")
)

Агрегируем рекомендации

In [21]:
recommendations = (
    candidate_books
    .groupBy("ISBN")
    .agg(
        F.countDistinct("User-ID").alias("liked_by_neighbors"),
        F.avg("Book-Rating").alias("avg_neighbor_rating")
    )
)

Добавляем информацию о книгах

In [22]:
result = (
    recommendations
    .join(
        books.select(
            "ISBN",
            "Book-Title",
            "Book-Author"
        ),
        on="ISBN",
        how="left"
    )
    .select(
        "ISBN",
        "Book-Title",
        "Book-Author",
        "liked_by_neighbors",
        "avg_neighbor_rating"
    )
    .orderBy(
        F.desc("liked_by_neighbors"),
        F.desc("avg_neighbor_rating")
    )
)

In [23]:
result.show(20, truncate=False)

+----------+------------------------------------------------------------------------------------------------------------------+----------------------+------------------+-------------------+
|ISBN      |Book-Title                                                                                                        |Book-Author           |liked_by_neighbors|avg_neighbor_rating|
+----------+------------------------------------------------------------------------------------------------------------------+----------------------+------------------+-------------------+
|0394704681|The Wisdom of Insecurity                                                                                          |Alan Wilson Watts     |1                 |10.0               |
|0345342968|Fahrenheit 451                                                                                                    |RAY BRADBURY          |1                 |10.0               |
|0060900075|The Doors of Perception and Heaven and

Классический user-based collaborative filtering можно реализовать средствами Spark DataFrame через self-join по общим объектам и последующий расчёт сходства пользователей. Однако такой подход плохо масштабируется: при большом количестве пользователей число потенциальных пар пользователей и объём shuffle быстро растут. Поэтому в Spark ML для collaborative filtering используется ALS — модельный метод матричной факторизации, который лучше приспособлен для распределённых вычислений. Для общего приближённого поиска ближайших соседей Spark также предоставляет LSH, однако это не специализированный user-user recommender.

# Обучение

Берём только явные оценки

In [24]:
ratings_explicit = (
    ratings
    .filter(F.col("Book-Rating") > 0)
    .select(
        F.col("User-ID").cast("int").alias("user_id"),
        F.col("ISBN"),
        F.col("Book-Rating").cast("float").alias("rating")
    )
)

In [25]:
ratings_explicit.show(5)

+-------+----------+------+
|user_id|      ISBN|rating|
+-------+----------+------+
| 276726|0155061224|   5.0|
| 276729|052165615X|   3.0|
| 276729|0521795028|   6.0|
| 276736|3257224281|   8.0|
| 276737|0600570967|   6.0|
+-------+----------+------+
only showing top 5 rows



Преобразуем ISBN в числовой ID

In [26]:
isbn_indexer = StringIndexer(
    inputCol="ISBN",
    outputCol="item_id"
)

In [27]:
isbn_model = isbn_indexer.fit(ratings_explicit)

In [28]:
als_data = (
    isbn_model
    .transform(ratings_explicit)
    .select(
        "user_id",
        F.col("item_id").cast("int").alias("item_id"),
        "rating",
        "ISBN"
    )
)

In [29]:
als_data.show(5)

+-------+-------+------+----------+
|user_id|item_id|rating|      ISBN|
+-------+-------+------+----------+
| 276726|  67111|   5.0|0155061224|
| 276729|  98858|   3.0|052165615X|
| 276729|  98875|   6.0|0521795028|
| 276736|  19512|   8.0|3257224281|
| 276737| 105804|   6.0|0600570967|
+-------+-------+------+----------+
only showing top 5 rows



Train/test split

In [30]:
train, test = als_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

In [31]:
print("Train:", train.count())
print("Test:", test.count())

Train: 346954
Test: 86717


Настраиваем ALS

In [32]:
als = ALS(
    userCol="user_id",
    itemCol="item_id",
    ratingCol="rating",

    rank=20,
    maxIter=10,
    regParam=0.1,

    implicitPrefs=False,
    nonnegative=True,

    coldStartStrategy="drop",
    seed=42
)

Обучаем

In [33]:
als_model = als.fit(train)

In [34]:
predictions = als_model.transform(test)

In [35]:
predictions.select(
    "user_id",
    "item_id",
    "rating",
    "prediction"
).show(20)

+-------+-------+------+----------+
|user_id|item_id|rating|prediction|
+-------+-------+------+----------+
|  48046|     26|   9.0| 6.8385763|
|   2891|     26|   7.0|  7.479297|
|  11676|     26|   1.0|  5.640265|
|  24379|     26|   8.0| 6.2866035|
|  25403|     26|   5.0| 4.8048677|
|  88146|     26|   8.0|  8.925162|
|  90517|     26|   8.0|  5.746448|
|  98051|     26|   2.0| 5.9347873|
|  88187|     26|   9.0|  8.639031|
| 104636|     26|   8.0|  8.276037|
|  88693|     26|   7.0| 6.8966084|
| 100846|     26|   8.0|  6.602677|
| 111954|     26|   5.0| 5.4913797|
| 112001|     26|   9.0| 8.5980835|
| 126671|     26|   6.0| 7.5414486|
| 115003|     26|   8.0|  7.199059|
| 116582|     26|  10.0|   7.42198|
| 121817|     26|   8.0|  6.160096|
| 108155|     26|   7.0| 5.5766306|
| 166062|     26|   8.0|  6.782285|
+-------+-------+------+----------+
only showing top 20 rows



RMSE

In [36]:
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

In [37]:
rmse = evaluator.evaluate(predictions)

In [38]:
print(f"RMSE: {rmse:.4f}")

RMSE: 2.5118


RMSE не является полной оценкой качества рекомендательной системы. Эта метрика оценивает точность прогнозирования рейтинга, но непосредственно не показывает, насколько хорошо система формирует итоговый top-N список рекомендаций. Для пользователя важнее, чтобы наиболее релевантные книги находились в верхней части списка. Поэтому для более полной оценки рекомендательных систем могут дополнительно применяться ranking-метрики.

# Рекомендации

In [39]:
user_recs = als_model.recommendForAllUsers(10)

In [40]:
user_recs.show(5, truncate=False)

+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                                 |
+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|12     |[{155513, 9.976739}, {126855, 8.538966}, {156001, 7.852236}, {157869, 7.579944}, {147086, 7.5018106}, {67856, 7.447414}, {133564, 7.399866}, {33665, 7.3655667}, {147106, 7.3165956}, {181727, 7.2396817}]      |
|26     |[{25588, 12.109985}, {29438, 12.019064}, {94097, 11.933226}, {33068, 11.87616}, {163345, 11.726783}, {54260, 11.726

In [41]:
recs_flat = (
    user_recs
    .select(
        "user_id",
        F.explode("recommendations").alias("rec")
    )
    .select(
        "user_id",
        F.col("rec.item_id").alias("item_id"),
        F.col("rec.rating").alias("predicted_rating")
    )
)

In [42]:
recs_flat.show(20)

+-------+-------+----------------+
|user_id|item_id|predicted_rating|
+-------+-------+----------------+
|     12| 155513|        9.976739|
|     12| 126855|        8.538966|
|     12| 156001|        7.852236|
|     12| 157869|        7.579944|
|     12| 147086|       7.5018106|
|     12|  67856|        7.447414|
|     12| 133564|        7.399866|
|     12|  33665|       7.3655667|
|     12| 147106|       7.3165956|
|     12| 181727|       7.2396817|
|     26|  25588|       12.109985|
|     26|  29438|       12.019064|
|     26|  94097|       11.933226|
|     26|  33068|        11.87616|
|     26| 163345|       11.726783|
|     26|  54260|       11.726783|
|     26|  54056|       11.722416|
|     26|  53443|       11.592546|
|     26|  89378|       11.582173|
|     26|  65607|       11.582173|
+-------+-------+----------------+
only showing top 20 rows



In [43]:
item_mapping = (
    als_data
    .select("item_id", "ISBN")
    .distinct()
)

In [44]:
recs_with_isbn = (
    recs_flat
    .join(
        item_mapping,
        on="item_id",
        how="left"
    )
)

In [45]:
recs_with_books = (
    recs_with_isbn
    .join(
        books.select(
            "ISBN",
            "Book-Title",
            "Book-Author"
        ),
        on="ISBN",
        how="left"
    )
    .select(
        "user_id",
        "ISBN",
        "Book-Title",
        "Book-Author",
        "predicted_rating"
    )
)

In [46]:
recs_with_books.show(20, truncate=False)

+-------+----------+-----------------------------------------------------------------+----------------------------+----------------+
|user_id|ISBN      |Book-Title                                                       |Book-Author                 |predicted_rating|
+-------+----------+-----------------------------------------------------------------+----------------------------+----------------+
|19     |0099698900|NULL                                                             |NULL                        |9.476887        |
|17     |0099698900|NULL                                                             |NULL                        |8.185706        |
|182    |0140441190|Queen of Spades and Other Stories (Classics S.)                  |Aleksandr Sergeevich Pushkin|7.755954        |
|70     |0425065979|A Visit from the Footbinder                                      |Emily Prager                |7.8986015       |
|92     |055010206X|NULL                                             

Для одного пользователя

In [47]:
target_user = (
    train
    .groupBy("user_id")
    .agg(F.count("*").alias("ratings_count"))
    .orderBy(F.desc("ratings_count"))
    .first()["user_id"]
)

In [48]:
train.filter(
    F.col("user_id") == target_user
).show()

+-------+-------+------+----------+
|user_id|item_id|rating|      ISBN|
+-------+-------+------+----------+
|  11676|      1|   6.0|0971880107|
|  11676|      2|   9.0|0385504209|
|  11676|      3|  10.0|0312195516|
|  11676|      6|  10.0|059035342X|
|  11676|      8|  10.0|0446672211|
|  11676|     13|  10.0|067976402X|
|  11676|     15|   9.0|0786868716|
|  11676|     16|   8.0|0743418174|
|  11676|     17|   6.0|0345337662|
|  11676|     19|   6.0|0156027321|
|  11676|     20|   6.0|0312278586|
|  11676|     23|   7.0|0440226430|
|  11676|     25|   1.0|044021145X|
|  11676|     28|   6.0|0446605239|
|  11676|     29|   8.0|0345370775|
|  11676|     30|   9.0|0385484518|
|  11676|     31|   8.0|0440241073|
|  11676|     32|  10.0|0345417623|
|  11676|     38|  10.0|0345361792|
|  11676|     39|  10.0|0446610038|
+-------+-------+------+----------+
only showing top 20 rows



In [49]:
user_df = spark.createDataFrame(
    [(target_user,)],
    ["user_id"]
)

In [50]:
user_recs = als_model.recommendForUserSubset(
    user_df,
    10
)

In [51]:
user_recs.show(truncate=False)

+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                               |
+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|11676  |[{57877, 12.937519}, {53093, 11.8156395}, {161454, 11.386302}, {154542, 11.354992}, {81627, 11.333843}, {158950, 11.328853}, {158949, 11.328853}, {179571, 11.324379}, {131167, 11.28935}, {90568, 11.253069}]|
+-------+---------------------------------------------------------------------------------------------------------------------------

In [52]:
new_user_id = 999999999

In [53]:
new_user = spark.createDataFrame(
    [(999999999,)],
    ["user_id"]
)

In [54]:
als_model.recommendForUserSubset(
    new_user,
    10
).show(truncate=False)

+-------+---------------+
|user_id|recommendations|
+-------+---------------+
+-------+---------------+



Персональных рекомендаций для него ALS построить не сможет. Причина принципиальная: во время обучения ALS построил факторный вектор только для пользователей, встречавшихся в train.

Достаточно ли одной оценки в training set? Технически — да. Качественно — обычно нет. Если пользователь присутствует в обучающей выборке хотя бы с одной корректной оценкой известной книге, ALS может построить для него пользовательский фактор и затем выдавать рекомендации.

Подключаем текстовые данные о книгах

In [55]:
item_mapping = (
    als_data
    .select("item_id", "ISBN")
    .distinct()
)

In [56]:
user_recs = als_model.recommendForAllUsers(10)

In [57]:
user_recs.show(5, truncate=False)

+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                                 |
+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|12     |[{155513, 9.976739}, {126855, 8.538966}, {156001, 7.852236}, {157869, 7.579944}, {147086, 7.5018106}, {67856, 7.447414}, {133564, 7.399866}, {33665, 7.3655667}, {147106, 7.3165956}, {181727, 7.2396817}]      |
|26     |[{25588, 12.109985}, {29438, 12.019064}, {94097, 11.933226}, {33068, 11.87616}, {163345, 11.726783}, {54260, 11.726

In [58]:
recs_flat = (
    user_recs
    .select(
        "user_id",
        F.explode("recommendations").alias("rec")
    )
    .select(
        "user_id",
        F.col("rec.item_id").alias("item_id"),
        F.col("rec.rating").alias("predicted_rating")
    )
)

In [59]:
recs_flat.show(10)

+-------+-------+----------------+
|user_id|item_id|predicted_rating|
+-------+-------+----------------+
|     12| 155513|        9.976739|
|     12| 126855|        8.538966|
|     12| 156001|        7.852236|
|     12| 157869|        7.579944|
|     12| 147086|       7.5018106|
|     12|  67856|        7.447414|
|     12| 133564|        7.399866|
|     12|  33665|       7.3655667|
|     12| 147106|       7.3165956|
|     12| 181727|       7.2396817|
+-------+-------+----------------+
only showing top 10 rows



Возвращаем ISBN

In [60]:
recs_with_isbn = (
    recs_flat
    .join(
        item_mapping,
        on="item_id",
        how="left"
    )
)

Добавляем название, автора и другие данные

In [61]:
recs_with_books = (
    recs_with_isbn
    .join(
        books.select(
            "ISBN",
            "Book-Title",
            "Book-Author",
            "Year-Of-Publication",
            "Publisher"
        ),
        on="ISBN",
        how="left"
    )
)

In [62]:
recs_with_books.select(
    "user_id",
    "Book-Title",
    "Book-Author",
    "Year-Of-Publication",
    "Publisher",
    "predicted_rating"
).show(20, truncate=False)

+-------+-----------------------------------------------------------------+----------------------------+-------------------+------------------------+----------------+
|user_id|Book-Title                                                       |Book-Author                 |Year-Of-Publication|Publisher               |predicted_rating|
+-------+-----------------------------------------------------------------+----------------------------+-------------------+------------------------+----------------+
|19     |NULL                                                             |NULL                        |NULL               |NULL                    |9.476887        |
|17     |NULL                                                             |NULL                        |NULL               |NULL                    |8.185706        |
|182    |Queen of Spades and Other Stories (Classics S.)                  |Aleksandr Sergeevich Pushkin|1978               |Penguin Books           |7.755954        

Для одного конкретного пользователя

In [63]:
target_user = (
    train
    .groupBy("user_id")
    .count()
    .orderBy(F.desc("count"))
    .first()["user_id"]
)

In [64]:
user_df = spark.createDataFrame(
    [(target_user,)],
    ["user_id"]
)

In [65]:
target_recs = als_model.recommendForUserSubset(
    user_df,
    10
)

In [66]:
target_recs_flat = (
    target_recs
    .select(
        "user_id",
        F.explode("recommendations").alias("rec")
    )
    .select(
        "user_id",
        F.col("rec.item_id").alias("item_id"),
        F.col("rec.rating").alias("predicted_rating")
    )
)

In [67]:
target_result = (
    target_recs_flat
    .join(item_mapping, "item_id")
    .join(
        books.select(
            "ISBN",
            "Book-Title",
            "Book-Author",
            "Year-Of-Publication",
            "Publisher"
        ),
        "ISBN",
        "left"
    )
    .select(
        "Book-Title",
        "Book-Author",
        "Year-Of-Publication",
        "Publisher",
        "predicted_rating"
    )
    .orderBy(F.desc("predicted_rating"))
)

In [68]:
target_result.show(10, truncate=False)

+-------------------------------------------------------------------------+-----------------------+-------------------+-------------------------+----------------+
|Book-Title                                                               |Book-Author            |Year-Of-Publication|Publisher                |predicted_rating|
+-------------------------------------------------------------------------+-----------------------+-------------------+-------------------------+----------------+
|Writer's Choice Â© 2001 Grade 7 Student Edition : Grammar and Composition|Royster                |2000               |Glencoe/McGraw-Hill      |12.937519       |
|Faust - Der Tragodie Erster Teil                                         |Goethe                 |0                  |Ernst Klett Verlag GmbH  |11.8156395      |
|NULL                                                                     |NULL                   |NULL               |NULL                     |11.386302       |
|Eating Fire: Selected

# Гибридная система

Топ популярных книг для cold-start

Популярность определим как число явных оценок, а при равенстве — средний рейтинг.

In [69]:
popular_top3 = (
    book_popularity
    .join(
        books.select(
            "ISBN",
            "Book-Title",
            "Book-Author"
        ),
        on="ISBN",
        how="left"
    )
    .filter(F.col("Book-Title").isNotNull())
    .select(
        "ISBN",
        "Book-Title",
        "Book-Author",
        "users_count",
        "avg_rating"
    )
    .orderBy(
        F.desc("users_count"),
        F.desc("avg_rating")
    )
    .limit(3)
)

In [70]:
popular_top3.show(truncate=False)

+----------+-------------------------+------------+-----------+------------------+
|ISBN      |Book-Title               |Book-Author |users_count|avg_rating        |
+----------+-------------------------+------------+-----------+------------------+
|0971880107|Wild Animus              |Rich Shapero|2502       |1.0195843325339728|
|0316666343|The Lovely Bones: A Novel|Alice Sebold|1295       |4.468725868725869 |
|0385504209|The Da Vinci Code        |Dan Brown   |883        |4.652321630804077 |
+----------+-------------------------+------------+-----------+------------------+



Проверяем, знает ли ALS пользователя

In [71]:
als_model.userFactors.show(5)

+---+--------------------+
| id|            features|
+---+--------------------+
| 70|[0.4890916, 0.0, ...|
|110|[0.24121003, 0.26...|
|160|[0.0, 0.60660434,...|
|190|[0.0, 0.0, 0.0, 0...|
|230|[0.305009, 0.2493...|
+---+--------------------+
only showing top 5 rows



In [72]:
def user_is_known(user_id):
    return (
        als_model.userFactors
        .filter(F.col("id") == int(user_id))
        .limit(1)
        .count() > 0
    )

Рекомендации старому пользователю через ALS

In [98]:
def als_top3(user_id):

    user_df = spark.createDataFrame(
        [(int(user_id),)],
        ["user_id"]
    )

    recs = als_model.recommendForUserSubset(
        user_df,
        50
    )

    recs_flat = (
        recs
        .select(
            "user_id",
            F.explode("recommendations").alias("rec")
        )
        .select(
            "user_id",
            F.col("rec.item_id").alias("item_id"),
            F.col("rec.rating").alias("score")
        )
    )

    seen_items = (
        train
        .filter(F.col("user_id") == int(user_id))
        .select("item_id")
        .distinct()
    )

    result = (
        recs_flat
        .join(
            seen_items,
            on="item_id",
            how="left_anti"
        )
        .join(
            item_mapping,
            on="item_id",
            how="left"
        )
        .join(
            books.select(
                "ISBN",
                "Book-Title",
                "Book-Author"
            ),
            on="ISBN",
            how="left"
        )
        .filter(F.col("Book-Title").isNotNull())
        .select(
            "user_id",
            "ISBN",
            "Book-Title",
            "Book-Author",
            "score"
        )
        .orderBy(F.desc("score"))
        .limit(3)
    )

    return result

In [99]:
old_user = (
    train
    .groupBy("user_id")
    .count()
    .orderBy(F.desc("count"))
    .first()["user_id"]
)

In [100]:
als_top3(old_user).show(truncate=False)

+-------+----------+-------------------------------------------------------------------------+-----------------------+----------+
|user_id|ISBN      |Book-Title                                                               |Book-Author            |score     |
+-------+----------+-------------------------------------------------------------------------+-----------------------+----------+
|11676  |0028181484|Writer's Choice Â© 2001 Grade 7 Student Edition : Grammar and Composition|Royster                |12.937519 |
|11676  |3123512304|Faust - Der Tragodie Erster Teil                                         |Goethe                 |11.8156395|
|11676  |1860495052|Eating Fire: Selected Poetry 1965-1995                                   |Margaret Eleanor Atwood|11.354992 |
+-------+----------+-------------------------------------------------------------------------+-----------------------+----------+



Cold user → популярные книги

In [76]:
def popular_fallback_top3():
    return (
        book_popularity
        .join(
            books.select(
                "ISBN",
                "Book-Title",
                "Book-Author"
            ),
            on="ISBN",
            how="left"
        )
        .filter(F.col("Book-Title").isNotNull())
        .select(
            "ISBN",
            "Book-Title",
            "Book-Author",
            "users_count",
            "avg_rating"
        )
        .orderBy(
            F.desc("users_count"),
            F.desc("avg_rating")
        )
        .limit(3)
    )

In [77]:
def hybrid_recommend(user_id):

    if user_is_known(user_id):
        print("Пользователь известен -> ALS")
        return als_top3(user_id)

    print("Холодный пользователь -> популярные книги")
    return popular_fallback_top3()

In [78]:
hybrid_recommend(old_user).show(
    truncate=False
)

Пользователь известен -> ALS
+-------+----------+-------------------------------------------------------------------------+-----------+----------+
|user_id|ISBN      |Book-Title                                                               |Book-Author|score     |
+-------+----------+-------------------------------------------------------------------------+-----------+----------+
|11676  |3123512304|Faust - Der Tragodie Erster Teil                                         |Goethe     |11.8156395|
|11676  |0028181484|Writer's Choice Â© 2001 Grade 7 Student Edition : Grammar and Composition|Royster    |12.937519 |
|11676  |2881688942|NULL                                                                     |NULL       |11.386302 |
+-------+----------+-------------------------------------------------------------------------+-----------+----------+



In [79]:
hybrid_recommend(999999999).show(
    truncate=False
)

Холодный пользователь -> популярные книги
+----------+-------------------------+------------+-----------+------------------+
|ISBN      |Book-Title               |Book-Author |users_count|avg_rating        |
+----------+-------------------------+------------+-----------+------------------+
|0971880107|Wild Animus              |Rich Shapero|2502       |1.0195843325339728|
|0316666343|The Lovely Bones: A Novel|Alice Sebold|1295       |4.468725868725869 |
|0385504209|The Da Vinci Code        |Dan Brown   |883        |4.652321630804077 |
+----------+-------------------------+------------+-----------+------------------+



Но cold-start можно сделать гораздо умнее. Умный cold-start через любимых авторов.

Допустим, высокой считаем оценку ≥ 8.

In [80]:
NEW_USER = 999999999

In [81]:
new_user_ratings = spark.createDataFrame(
    [
        (NEW_USER, "0345339681", 9.0),
        (NEW_USER, "0439064872", 8.0)
    ],
    ["user_id", "ISBN", "rating"]
)

Получаем любимых авторов

In [82]:
liked_authors = (
    new_user_ratings
    .filter(F.col("rating") >= 8)
    .join(
        books.select(
            "ISBN",
            "Book-Author"
        ),
        on="ISBN",
        how="inner"
    )
    .select("Book-Author")
    .distinct()
)

In [83]:
liked_authors.show()

+--------------+
|   Book-Author|
+--------------+
|J.R.R. TOLKIEN|
| J. K. Rowling|
+--------------+



Теперь найдём другие книги этих авторов

In [84]:
already_seen = (
    new_user_ratings
    .select("ISBN")
    .distinct()
)

In [85]:
content_candidates = (
    books
    .join(
        liked_authors,
        on="Book-Author",
        how="inner"
    )
    .join(
        already_seen,
        on="ISBN",
        how="left_anti"
    )
)

Затем можно использовать глобальную популярность только для ранжирования этих уже персонализированных кандидатов

In [86]:
smart_cold_top3 = (
    content_candidates
    .join(
        book_popularity,
        on="ISBN",
        how="left"
    )
    .fillna({
        "users_count": 0,
        "avg_rating": 0.0
    })
    .orderBy(
        F.desc("users_count"),
        F.desc("avg_rating")
    )
    .select(
        "ISBN",
        "Book-Title",
        "Book-Author",
        "users_count",
        "avg_rating"
    )
    .limit(3)
)

In [87]:
smart_cold_top3.show(truncate=False)

+----------+----------------------------------------------------------------+--------------+-----------+-----------------+
|ISBN      |Book-Title                                                      |Book-Author   |users_count|avg_rating       |
+----------+----------------------------------------------------------------+--------------+-----------+-----------------+
|059035342X|Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))|J. K. Rowling |571        |4.900175131348512|
|043935806X|Harry Potter and the Order of the Phoenix (Book 5)              |J. K. Rowling |334        |5.57185628742515 |
|0345339703|The Fellowship of the Ring (The Lord of the Rings, Part 1)      |J.R.R. TOLKIEN|257        |4.505836575875486|
+----------+----------------------------------------------------------------+--------------+-----------+-----------------+



Таким образом, новый пользователь уже не получает просто глобальный top-3. Он получает популярные книги тех авторов, которые ему уже понравились.

Можно использовать несколько признаков книги. вашем Books.csv есть, в частности Book-Title, Book-Author, Year-Of-Publication, Publisher.
Поэтому профиль пользователя можно строить не только по автору. Например, если пользователь высоко оценивал книги одного издательства.

## Единое гибридное решение

Определим три состояния пользователя:
1. Старый пользователь -> ALS
2. Новый пользователь, но уже есть оценки -> content-based + popularity
3. Совершенно новый пользователь -> popularity

In [88]:
def hybrid_top3(user_id, recent_ratings=None):

    # 1. Старый пользователь -> ALS
    if user_is_known(user_id):
        print("Strategy: ALS")
        return als_top3(user_id)

    # 2. Новый пользователь, но уже есть оценки
    if recent_ratings is not None:

        has_ratings = (
            recent_ratings
            .filter(F.col("user_id") == user_id)
            .limit(1)
            .count() > 0
        )

        if has_ratings:

            print("Strategy: Content + Popularity")

            user_ratings = (
                recent_ratings
                .filter(F.col("user_id") == user_id)
            )

            # Авторы книг, которым пользователь поставил >= 8
            liked_authors = (
                user_ratings
                .filter(F.col("rating") >= 8)
                .join(
                    books.select(
                        "ISBN",
                        "Book-Author"
                    ),
                    on="ISBN",
                    how="inner"
                )
                .select("Book-Author")
                .filter(F.col("Book-Author").isNotNull())
                .distinct()
            )

            # Если нет ни одной высокой оценки,
            # используем обычную популярность
            if liked_authors.limit(1).count() == 0:
                return popular_fallback_top3()

            # Уже оценённые книги
            seen = (
                user_ratings
                .select("ISBN")
                .distinct()
            )

            result = (
                books
                .join(
                    liked_authors,
                    on="Book-Author",
                    how="inner"
                )
                .join(
                    seen,
                    on="ISBN",
                    how="left_anti"
                )
                .join(
                    book_popularity,
                    on="ISBN",
                    how="left"
                )
                .fillna({
                    "users_count": 0,
                    "avg_rating": 0.0
                })
                .filter(F.col("Book-Title").isNotNull())
                .select(
                    "ISBN",
                    "Book-Title",
                    "Book-Author",
                    "users_count",
                    "avg_rating"
                )
                .orderBy(
                    F.desc("users_count"),
                    F.desc("avg_rating")
                )
                .limit(3)
            )

            # Если по авторам кандидатов не нашли
            if result.limit(1).count() == 0:
                return popular_fallback_top3()

            return result

    # 3. Полностью новый пользователь
    print("Strategy: Popularity")
    return popular_fallback_top3()

Старый пользователь

In [89]:
old_user = (
    train
    .groupBy("user_id")
    .count()
    .orderBy(F.desc("count"))
    .first()["user_id"]
)

In [90]:
print("Старый пользователь:", old_user)

Старый пользователь: 11676


In [101]:
result_old = hybrid_top3(old_user)

Strategy: ALS


In [102]:
result_old.show(
    3,
    truncate=False
)

+-------+----------+-------------------------------------------------------------------------+-----------------------+----------+
|user_id|ISBN      |Book-Title                                                               |Book-Author            |score     |
+-------+----------+-------------------------------------------------------------------------+-----------------------+----------+
|11676  |0028181484|Writer's Choice Â© 2001 Grade 7 Student Edition : Grammar and Composition|Royster                |12.937519 |
|11676  |3123512304|Faust - Der Tragodie Erster Teil                                         |Goethe                 |11.8156395|
|11676  |1860495052|Eating Fire: Selected Poetry 1965-1995                                   |Margaret Eleanor Atwood|11.354992 |
+-------+----------+-------------------------------------------------------------------------+-----------------------+----------+



Полностью новый пользователь

In [93]:
new_user = 999999999

result_new = hybrid_top3(new_user)

result_new.show(
    3,
    truncate=False
)

Strategy: Popularity
+----------+-------------------------+------------+-----------+------------------+
|ISBN      |Book-Title               |Book-Author |users_count|avg_rating        |
+----------+-------------------------+------------+-----------+------------------+
|0971880107|Wild Animus              |Rich Shapero|2502       |1.0195843325339728|
|0316666343|The Lovely Bones: A Novel|Alice Sebold|1295       |4.468725868725869 |
|0385504209|The Da Vinci Code        |Dan Brown   |883        |4.652321630804077 |
+----------+-------------------------+------------+-----------+------------------+



Новый пользователь, который уже что-то оценил

In [94]:
books.select(
    "ISBN",
    "Book-Title",
    "Book-Author"
).show(10, truncate=False)

+----------+--------------------------------------------------------------------------------------------------+--------------------+
|ISBN      |Book-Title                                                                                        |Book-Author         |
+----------+--------------------------------------------------------------------------------------------------+--------------------+
|0195153448|Classical Mythology                                                                               |Mark P. O. Morford  |
|0002005018|Clara Callan                                                                                      |Richard Bruce Wright|
|0060973129|Decision in Normandy                                                                              |Carlo D'Este        |
|0374157065|Flu: The Story of the Great Influenza Pandemic of 1918 and the Search for the Virus That Caused It|Gina Bari Kolata    |
|0393045218|The Mummies of Urumchi                                   

In [95]:
new_user_ratings = spark.createDataFrame(
    [
        (
            999999999,
            "0345339681",
            9.0
        ),
        (
            999999999,
            "0439064872",
            8.0
        )
    ],
    [
        "user_id",
        "ISBN",
        "rating"
    ]
)

In [96]:
result_smart_new = hybrid_top3(
    999999999,
    new_user_ratings
)

Strategy: Content + Popularity


In [97]:
result_smart_new.show(
    3,
    truncate=False
)

+----------+----------------------------------------------------------------+--------------+-----------+-----------------+
|ISBN      |Book-Title                                                      |Book-Author   |users_count|avg_rating       |
+----------+----------------------------------------------------------------+--------------+-----------+-----------------+
|059035342X|Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))|J. K. Rowling |571        |4.900175131348512|
|043935806X|Harry Potter and the Order of the Phoenix (Book 5)              |J. K. Rowling |334        |5.57185628742515 |
|0345339703|The Fellowship of the Ring (The Lord of the Rings, Part 1)      |J.R.R. TOLKIEN|257        |4.505836575875486|
+----------+----------------------------------------------------------------+--------------+-----------+-----------------+

